# 03 SpaceX Falcon 9 - Data Wrangling & Target Label Engineering

This notebook is Step 03 in a multi-step end-to-end data science project that analyzes and predicts first-stage landing success for SpaceX Falcon 9 launches. It converts the launch-level dataset produced in **01_data_collection_api.ipynb** into an ML-ready table by:

- running a quick quality check (shape, missingness, dtypes)

- validating a few key distributions (launch site, orbit, recorded landing outcome)

- creating a binary target label `Class` for **first-stage landing success**

**Pipeline overview:**

- **Step 01:** Collect launch data from the SpaceX REST API and create an initial modeling dataset.

- **Step 02:** Scrape a *fixed Wikipedia revision* of Falcon 9 & Falcon Heavy launch tables and export a clean CSV for supplementary analysis.

- **Step 03 (this notebook):** Clean and engineer features from the Step 01 dataset, create the landing success label, and export the modeling dataset used in downstream EDA and machine learning.

- **Next steps:** Exploratory analysis (SQL+ visualization), interactive geospatial mapping, dashboarding, and machine learning modeling.

**Input:** `../data/processed/01_dataset_part_1.csv` (Step 01)

**Output:** `../data/processed/03_dataset_part_2.csv`

## Notebook sections

**1. Setup**

**2. Load Step 01 dataset**

**3. Quick data audit**

**4. Lightweight distribution checks**

**5. Label engineering**

**6. Save artifacts**

**7. Assumptions**

---

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

# Paths
PROJECT_ROOT = Path('..').resolve()
DATA_DIR = PROJECT_ROOT / 'data'

RAW_DIR = DATA_DIR / 'raw'
RAW_DIR.mkdir(parents = True, exist_ok = True)

PROCESSED_DIR = DATA_DIR / 'processed'
PROCESSED_DIR.mkdir(parents = True, exist_ok = True)

## 2. Load Step 01 dataset

We load the curated launch dataset produced in **Step 01 (SpaceX API collection)**.

The goal here is making sure the dataset is consistent and then generating a reliable target label for modeling later.

In [2]:
input_path = PROCESSED_DIR / '01_dataset_part_1.csv'

df = pd.read_csv(input_path)
df.head()

,FlightNumber,Date,BoosterVersion,PayloadMass,Orbit,LaunchSite,Outcome,Flights,GridFins,Reused,Legs,LandingPad,Block,ReusedCount,Serial,Longitude,Latitude
0,1,2010-06-04,Falcon 9,6123.547647,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0003,-80.577366,28.561857
1,2,2012-05-22,Falcon 9,525.000000,LEO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0005,-80.577366,28.561857
2,3,2013-03-01,Falcon 9,677.000000,ISS,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B0007,-80.577366,28.561857
3,4,2013-09-29,Falcon 9,500.000000,PO,VAFB SLC 4E,False Ocean,1,False,False,False,NaN,1.0,0,B1003,-120.610829,34.632093
4,5,2013-12-03,Falcon 9,3170.000000,GTO,CCSFS SLC 40,None None,1,False,False,False,NaN,1.0,0,B1004,-80.577366,28.561857


## 3. Quick data audit

A minimal audit to catch obvious issues early (missing columns, unexpected nulls).

In [3]:
# Dataset shape
df.shape

(90, 17)

In [4]:
# Missingness overview
missing_pct = (df.isnull().mean() * 100).sort_values(ascending = False)
missing_pct[missing_pct > 0].round(2)

LandingPad    28.89
dtype: float64

In [5]:
# Column dtypes
df.dtypes

FlightNumber        int64
Date               object
BoosterVersion     object
PayloadMass       float64
Orbit              object
LaunchSite         object
Outcome            object
Flights             int64
GridFins             bool
Reused               bool
Legs                 bool
LandingPad         object
Block             float64
ReusedCount         int64
Serial             object
Longitude         float64
Latitude          float64
dtype: object

## 4. Distribution checks

These quick counts validate that the dataset looks reasonable and provide intuition for later EDA.

### 4.1 Launch frequency by site

In [6]:
df['LaunchSite'].value_counts()

LaunchSite
CCSFS SLC 40    55
KSC LC 39A      22
VAFB SLC 4E     13
Name: count, dtype: int64

### 4.2 Orbit frequency (excluding GTO)

In [7]:
df[df['Orbit'] != 'GTO']['Orbit'].value_counts()

Orbit
ISS      21
VLEO     14
PO        9
LEO       7
SSO       5
MEO       3
HEO       1
ES-L1     1
SO        1
GEO       1
Name: count, dtype: int64

### 4.3 Recorded first-stage landing outcomes

In [8]:
landing_outcomes = df['Outcome'].value_counts()
landing_outcomes

Outcome
True ASDS      41
None None      19
True RTLS      14
False ASDS      6
True Ocean      5
False Ocean     2
None ASDS       2
False RTLS      1
Name: count, dtype: int64

## 5. Label engineering

For supervised learning later in the project, we need a binary target label:

- `Class = 1` for outcomes where the first stage landed successfully (`True ASDS`, `True RTLS`, `True Ocean`)

- `Class = 0` for unsuccessful or unknown outcomes (`False*` and `None*`)

The mapping matches the original capstone lab logic, but is written explicitly to avoid brittle index-based selection.

In [9]:
# Outcomes considered 'not successful' (incl. unknown/none)
bad_outcomes = {
    'False ASDS',
    'False RTLS',
    'False Ocean',
    'None ASDS',
    'None RTLS',
    'None None',
}

# Determine which of the launches were successful or not
sorted(bad_outcomes.intersection(set(df['Outcome'].unique())))

['False ASDS', 'False Ocean', 'False RTLS', 'None ASDS', 'None None']

In [10]:
# Create a binary label for classifying landings
landing_class = []
for outcome in df['Outcome']:
    if outcome in bad_outcomes:
        landing_class.append(0)
    else:
        landing_class.append(1)

# Add the class to the dataframe
df['Class'] = landing_class

df[['Outcome', 'Class']].head(10)

,Outcome,Class
0,None None,0
1,None None,0
2,None None,0
3,False Ocean,0
4,None None,0
5,None None,0
6,True Ocean,1
7,True Ocean,1
8,None None,0
9,None None,0


## 6. Save artifacts

We save the labeled dataset for the downstream SQL / EDA / modeling notebooks.

In [11]:
output_path = PROCESSED_DIR / '03_dataset_part_2.csv'
df.to_csv(output_path, index = False)

## 7. Assumptions

- This notebook assumes `../data/processed/01_dataset_part_1.csv` exists and was produced by **01_data_collection_api.ipynb**.

- The `Outcome` field follows the capstone lab conventions (`True*`, `False*`, `None*`).

- We keep the original lab's binary label logic (anything not in the explicit *bad* list is treated as success).

---